# Prompt Optimizer - Evaluation

Compare base model vs fine-tuned model on test set.

**Setup:**
1. Change runtime type to **T4 GPU**
2. Run all cells

In [ ]:
#@title 1. Install & Load
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Change runtime type to T4 GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title 2. Load Models
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# Load base model
print("Loading base model...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

# Clone repo for LoRA adapters
import os
if not os.path.exists('prompt-optimizer'):
    !git clone --depth 1 https://github.com/Petsku01/prompt-optimizer.git

LORA_PATH = "prompt-optimizer/models/lora"
if os.path.exists(os.path.join(LORA_PATH, 'adapter_model.safetensors')):
    print("Loading LoRA adapters from repo...")
    from peft import PeftModel
    # Reload base for training mode first
    ft_model, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )
    ft_model = PeftModel.from_pretrained(ft_model, LORA_PATH)
    FastLanguageModel.for_inference(ft_model)
    print("LoRA adapters loaded!")
else:
    # Try local upload path
    alt_path = "prompt-optimizer-model/lora"
    if os.path.exists(os.path.join(alt_path, 'adapter_model.safetensors')):
        LORA_PATH = alt_path
        ft_model, _ = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
            dtype=None,
        )
        ft_model = PeftModel.from_pretrained(ft_model, LORA_PATH)
        FastLanguageModel.for_inference(ft_model)
        print("LoRA adapters loaded from local!")
    else:
        print("ERROR: No LoRA adapters found. Upload them or check repo.")
        ft_model = None

In [ ]:
#@title 3. Load Test Data
import json
import os

# Download test set from repo
TEST_PATH = "prompt-optimizer/data/test.jsonl"
if not os.path.exists(TEST_PATH):
    !git clone --depth 1 https://github.com/Petsku01/prompt-optimizer.git 2>/dev/null || true
    TEST_PATH = os.path.join("prompt-optimizer", "data", "test.jsonl")

test_data = []
with open(TEST_PATH) as f:
    for line in f:
        if line.strip():
            test_data.append(json.loads(line))

print(f"Test examples: {len(test_data)}")
print(f"Categories: {set(d['category'] for d in test_data)}")

SYSTEM_PROMPT = "You are a prompt engineering expert that transforms vague, underspecified prompts into clear, well-structured, and effective prompts. You preserve the user's original intent while adding relevant specificity, format constraints, audience targeting, and contextual details."
INSTRUCTION = "Optimize the following prompt to be clear, specific, and effective. Preserve the original intent while adding structure, context, and constraints where appropriate."

In [ ]:
#@title 4. Run Evaluation
import random
random.seed(42)

# Sample 20 diverse test examples (or all if <= 20)
sample_size = min(20, len(test_data))
sample = random.sample(test_data, sample_size)

def generate(model, tokenizer, vague_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{INSTRUCTION}\n\nPrompt to optimize: {vague_prompt}"},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

results = []
print(f"Evaluating {sample_size} examples on both models...\n")

for i, item in enumerate(sample):
    vague = item["vague"]
    expected = item["optimized"]
    category = item["category"]

    base_output = generate(base_model, tokenizer, vague)
    ft_output = generate(ft_model, tokenizer, vague) if ft_model else "(no LoRA)"

    results.append({
        "category": category,
        "vague": vague,
        "expected": expected[:200],
        "base_model": base_output,
        "finetuned": ft_output,
    })

    print(f"[{i+1}/{sample_size}] {category}: {vague[:40]}...")
    print(f"  BASE:      {base_output[:120]}")
    print(f"  FINETUNED: {ft_output[:120]}")
    print(f"  EXPECTED:  {expected[:120]}")
    print()

In [ ]:
#@title 5. Metrics & Results
import json

# Compute basic metrics
def compute_metrics(results):
    """Compute word overlap and length similarity metrics."""
    metrics = {"base": {}, "finetuned": {}}
    
    for model_key in ["base", "finetuned"]:
        model_name = f"{model_key}_model" if model_key == "base" else "finetuned"
        word_overlaps = []
        len_ratios = []
        has_structure = 0
        
        for r in results:
            output = r[model_name] if model_key == "base" else r["finetuned"]
            expected_words = set(r["expected"].lower().split())
            output_words = set(output.lower().split())
            
            # Word overlap (Jaccard)
            if expected_words and output_words:
                overlap = len(expected_words & output_words) / len(expected_words | output_words)
                word_overlaps.append(overlap)
            
            # Length ratio (output / expected)
            expected_len = len(r["expected"].split())
            output_len = len(output.split())
            if expected_len > 0:
                len_ratios.append(min(output_len / expected_len, 2.0))
            
            # Structure indicators (bullet points, numbered lists, colons)
            if any(c in output for c in ["-", "1.", ":", "\n"]):
                has_structure += 1
        
        metrics[model_key] = {
            "word_overlap": sum(word_overlaps) / len(word_overlaps) if word_overlaps else 0,
            "length_ratio": sum(len_ratios) / len(len_ratios) if len_ratios else 0,
            "structure_pct": has_structure / len(results) * 100,
            "avg_output_len": sum(len(r.get(model_name if model_key == "base" else "finetuned", "").split()) for r in results) / len(results),
        }
    
    return metrics

metrics = compute_metrics(results)

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"Test examples: {len(results)}")
print()
print(f"{'Metric':<20} {'Base Model':>12} {'Finetuned':>12}")
print("-" * 44)
print(f"{'Word overlap':<20} {metrics['base']['word_overlap']:>12.3f} {metrics['finetuned']['word_overlap']:>12.3f}")
print(f"{'Length ratio':<20} {metrics['base']['length_ratio']:>12.3f} {metrics['finetuned']['length_ratio']:>12.3f}")
print(f"{'Structure %':<20} {metrics['base']['structure_pct']:>11.1f}% {metrics['finetuned']['structure_pct']:>11.1f}%")
print(f"{'Avg output words':<20} {metrics['base']['avg_output_len']:>12.1f} {metrics['finetuned']['avg_output_len']:>12.1f}")

# Category breakdown
print()
print("Per-category structure rate:")
categories = set(r["category"] for r in results)
for cat in sorted(categories):
    cat_results = [r for r in results if r["category"] == cat]
    base_struct = sum(1 for r in cat_results if any(c in r["base_model"] for c in ["-", "1.", ":"])) / len(cat_results)
    ft_struct = sum(1 for r in cat_results if any(c in r["finetuned"] for c in ["-", "1.", ":"])) / len(cat_results)
    print(f"  {cat:<15} base={base_struct:.0%}  ft={ft_struct:.0%}")

# Save results
eval_output = {
    "metrics": metrics,
    "sample_size": len(results),
    "results": results,
}

with open("eval_results.json", "w") as f:
    json.dump(eval_output, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to eval_results.json")

In [ ]:
#@title 6. Detailed Comparison
print("DETAILED COMPARISON")
print("=" * 80)

for i, r in enumerate(results):
    print(f"\n{'─' * 80}")
    print(f"[{r['category'].upper()}] {r['vague']}")
    print(f"\nBASE MODEL:")
    print(f"  {r['base_model'][:300]}")
    print(f"\nFINETUNED:")
    print(f"  {r['finetuned'][:300]}")
    print(f"\nEXPECTED:")
    print(f"  {r['expected'][:300]}")